In [23]:
import pandas as pd
from textblob import TextBlob

In [24]:
df = pd.read_csv('../data/comments.csv')
df2 = pd.read_csv('../data/statistics.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   video_id  143 non-null    object
 1   comments  142 non-null    object
 2   date      143 non-null    object
 3   time      143 non-null    object
dtypes: object(4)
memory usage: 4.6+ KB


In [25]:

def comments_sentiment(text):
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'
    
df['comments'] = df['comments'].apply(comments_sentiment)
df.to_csv('../data/comments_sentiments.csv', index=False)
display(df)

,video_id,comments,date,time
0,Wmy5TQQPQho,neutral,2025-05-27,16:31:12
1,Wmy5TQQPQho,neutral,2025-05-27,16:31:12
2,Wmy5TQQPQho,negative,2025-05-27,16:31:12
3,Wmy5TQQPQho,neutral,2025-05-27,16:31:12
4,Wmy5TQQPQho,negative,2025-05-27,16:31:12
...,...,...,...,...
138,HAN2BYH53d8,positive,2024-03-23,17:37:01
139,HAN2BYH53d8,positive,2024-03-23,17:37:01
140,GHJOAA5Trh8,neutral,2025-05-11,08:44:43
141,GHJOAA5Trh8,positive,2025-05-11,08:44:43


In [26]:
df3 = pd.read_csv('../data/comments_sentiments.csv')

df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   video_id  143 non-null    object
 1   comments  143 non-null    object
 2   date      143 non-null    object
 3   time      143 non-null    object
dtypes: object(4)
memory usage: 4.6+ KB


In [27]:
def sentiment_percentages(group):
    total = len(group)
    positive = (group == 'positive').sum() / total * 100
    negative = (group == 'negative').sum() / total * 100
    neutral = (group == 'neutral').sum() / total * 100
    return pd.Series({
        'positive_pct': positive,
        'negative_pct': negative,
        'neutral_pct': neutral,
        'total_comments': total
    })


sentiment_summary = df3.groupby('video_id')['comments'].apply(sentiment_percentages).reset_index()

df_merged = df2.merge(sentiment_summary, on='video_id', how='left')

df_merged['engagement_stats'] = (
    df_merged['viewCount'] * 0.1 + df_merged['likeCount'] * 1 + df_merged['commentCount'] * 2
)

df_wide = df_merged.pivot_table(
    index=['video_id', 'channel_title', 'title', 'viewCount', 'likeCount', 'commentCount', 'date', 'time'],
    columns='level_1',
    values='comments'
).reset_index()

cleaned_sentiment = df_wide[df_wide['total_comments'] > 1.0]
cleaned_sentiment = df_wide.drop(columns=['channel_title', 'title', 'viewCount', 'likeCount', 'commentCount'])

cleaned_sentiment.to_csv('../data/sentiment.csv', index=False)
display(cleaned_sentiment.head(100))


level_1,video_id,date,time,negative_pct,neutral_pct,positive_pct,total_comments
0,-1ta-B1lA3s,2025-04-20,19:02:09,20.0,40.000000,40.000000,5.0
1,1Jjj_bitQyA,2020-06-30,03:00:00,0.0,80.000000,20.000000,5.0
2,46B-rfCAW4E,2025-05-07,16:31:18,0.0,100.000000,0.000000,5.0
3,7aJnShgDJcA,2024-12-17,17:58:32,0.0,80.000000,20.000000,5.0
4,CAzS_soXk4s,2025-05-15,16:53:54,40.0,60.000000,0.000000,5.0
5,E9j-Rf6aShI,2023-08-27,15:34:00,20.0,60.000000,20.000000,5.0
6,G5lS6O7j4RI,2025-05-30,17:48:58,20.0,80.000000,0.000000,5.0
7,GCp8X2o3tQc,2024-09-17,09:02:25,0.0,100.000000,0.000000,5.0
8,GHJOAA5Trh8,2025-05-11,08:44:43,0.0,33.333333,66.666667,3.0
9,HAN2BYH53d8,2024-03-23,17:37:01,0.0,60.000000,40.000000,5.0


In [28]:
df1 = pd.read_csv('../data/etc_comments.csv')
df2 = pd.read_csv('../data/comments_sentiments.csv')

df_merged_comment = pd.merge(df2, df1, on=['video_id'], how='left')
df_merged_comment = df_merged_comment.drop(columns=['date_x', 'time_x'])
df_merged_comment = df_merged_comment.rename(columns={'comments': 'sent_comment', 'date_y' : 'date', 'time_y' : 'time'})
df_merged_comment.to_csv('../data/detail-comment.csv', index=False)
df_merged_comment.head(2)

,video_id,sent_comment,comment_text,author,date,time
0,Wmy5TQQPQho,neutral,❤,@rockyjohnson2511,2025-05-31,18:29:32
1,Wmy5TQQPQho,neutral,Hài vc,@sungchusinh,2025-05-31,16:43:21
